# 03 · ¿Dónde se reduce o se invierte la prima?

**Objetivo:** Medir con precios reales cuánto sobreprecio sostiene la marca líder frente a su competidor directo, como caso de valor de marca para directores de marketing y ventas.

**Pregunta:** ¿En qué tiendas Coca-Cola queda cerca del precio de Pepsi o por debajo?

Importa porque ahí está la oportunidad de la marca retadora (o el riesgo de margen de la líder).

- **Invertida:** Coca-Cola más barata que Pepsi en esa tienda.
- **Reducida:** prima menor a la mitad de la mediana nacional.

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import entorno
entorno.setup()
import profeco, tablas, pares, estadistica as E, charts as C
import prima as P
PROC, META, CHARTS, SOCIAL = entorno.PROC, entorno.META, entorno.CHARTS, entorno.SOCIAL
CFG = entorno.config()
reg = entorno.registrar
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)
pts, st, cat = P.load()
REPS = 500
rows = []
for k in ["lata", "familiar"]:
    p = P.store_flags(pd.read_csv(PROC / f"pares_{k}.csv"))
    p.to_csv(PROC / f"pares_{k}_clase.csv", index=False, encoding="utf-8-sig")
    c = p.clase.value_counts(normalize=True).mul(100).reindex(["normal", "reducida", "igual", "invertida"]).fillna(0)
    rows.append(c.rename(P.COMPARA[k]["nombre"]))
cls = pd.DataFrame(rows)
display(cls.round(1))

clase,normal,reducida,igual,invertida
Lata 355 ml vs. lata 355 ml,95.7,2.7,1.1,0.5
Coca-Cola 2.75 L vs. Pepsi 3 L,75.0,4.2,0.0,20.8


## 1. ¿En qué formatos se concentra?

In [2]:
pf = pd.read_csv(PROC / "pares_familiar_clase.csv")
by = pf.groupby("formato").agg(tiendas=("clase", "size"), invertida=("clase", lambda s: (s == "invertida").mean() * 100),
                               reducida=("clase", lambda s: (s == "reducida").mean() * 100)).query("tiendas >= 10")
display(by.sort_values("invertida", ascending=False).round(0))
pl = pd.read_csv(PROC / "pares_lata_clase.csv")
display(pl[pl.clase != "normal"][["cadena_comercial", "municipio_id", "pesos_a", "pesos_b", "prima_pct", "clase"]].sort_values("prima_pct").head(12).round(1))

,tiendas,invertida,reducida
formato,,,
Bodega Aurrerá,27,63.0,11.0
Mega Soriana,15,33.0,0.0
Hipermercado Soriana,17,12.0,6.0
Chedraui,13,0.0,0.0
Walmart,30,0.0,0.0
Walmart Express,17,0.0,0.0


,cadena_comercial,municipio_id,pesos_a,pesos_b,prima_pct,clase
126,Wal-mart Express,Jalisco | Guadalajara,19.0,18.0,-5.3,invertida
121,Hipermercado Soriana,Guerrero | Acapulco de Juárez,21.5,21.5,0.0,igual
122,Mega Soriana,Guerrero | Acapulco de Juárez,21.5,21.5,0.0,igual
5,Chedraui,Campeche | Campeche,19.0,20.0,5.3,reducida
86,Bodega Aurrera,Estado de México | Ecatepec de Morelos,19.0,20.0,5.3,reducida
177,Bodega Aurrera,Veracruz | Veracruz,19.0,20.0,5.3,reducida
18,Hipermercado Soriana,Chihuahua | Juárez,19.6,20.8,5.9,reducida
1,Aprecio,Baja California | Tijuana,18.9,20.5,8.5,reducida


## 2. ¿Una tienda con prima baja en lata también la tiene en la botella?

Si fuera así, la tienda (o la cadena) estaría usando a Coca-Cola como gancho en general. Correlación de rangos entre la
prima de lata y la de botella en las tiendas que tienen las dos comparaciones.

In [3]:
m = pl[["establecimiento_id", "prima_pct", "formato"]].merge(pf[["establecimiento_id", "prima_pct"]], on="establecimiento_id",
                                                             suffixes=("_lata", "_familiar"))
rho = m[["prima_pct_lata", "prima_pct_familiar"]].corr(method="spearman").iloc[0, 1]
print(f"Tiendas con las dos comparaciones: {len(m)} · Spearman: {rho:+.2f}")

Tiendas con las dos comparaciones: 104 · Spearman: -0.04


In [4]:
cls.to_csv(PROC / "clases_prima.csv", encoding="utf-8-sig")
by.to_csv(PROC / "familiar_invertida_por_formato.csv", encoding="utf-8-sig")
F = "03_donde_se_reduce"
cl, cf = cls.iloc[0], cls.iloc[1]
reg("lata_invertida", cl.invertida, fuente=F); reg("lata_reducida", cl.reducida, fuente=F); reg("lata_igual", cl.igual, fuente=F)
reg("fam_invertida", cf.invertida, fuente=F); reg("fam_reducida", cf.reducida, fuente=F)
reg("fam_inv_max", by.invertida.max(), fuente=F, nota=by.invertida.idxmax())
reg("rho_lata_familiar", rho, n=len(m), fuente=F)
display(Markdown(
    f"**Conclusión.** En la lata la prima casi nunca se invierte ({cl.invertida:.0f}% de las tiendas) y se reduce a menos de la mitad en "
    f"{cl.reducida:.0f}%. En la botella familiar Coca-Cola sale más barata por litro en {cf.invertida:.0f}% de las tiendas, sobre todo en "
    f"{by.invertida.idxmax()} ({by.invertida.max():.0f}%). Las dos primas casi no se relacionan en la misma tienda (Spearman {rho:+.2f}): "
    "cada presentación tiene su propia política."))

**Conclusión.** En la lata la prima casi nunca se invierte (1% de las tiendas) y se reduce a menos de la mitad en 3%. En la botella familiar Coca-Cola sale más barata por litro en 21% de las tiendas, sobre todo en Bodega Aurrerá (63%). Las dos primas casi no se relacionan en la misma tienda (Spearman -0.04): cada presentación tiene su propia política.